# Подготовка данных

## Считаем данные в исходном csv формате

In [2]:
!pip install findspark

Defaulting to user installation because normal site-packages is not writeable


Перезапустить kernel после установки!

In [1]:
import findspark
findspark.init()

In [2]:
!hdfs dfs -ls

Found 1 items
drwxr-xr-x   - ubuntu hadoop          0 2025-02-11 10:34 data


In [3]:
!hdfs dfs -ls data

Found 3 items
-rw-r--r--   1 ubuntu hadoop       9703 2025-02-11 10:32 data/lectures.csv
-rw-r--r--   1 ubuntu hadoop     296161 2025-02-11 10:32 data/questions.csv
-rw-r--r--   1 ubuntu hadoop 5846760913 2025-02-11 10:34 data/train.csv


Создадим SparkSession

In [5]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
        .builder
        .appName("OTUS")
        .getOrCreate()
)

Загрузим данные

In [6]:
df = spark.read.csv("data/train.csv", inferSchema=True, header=True)

Поделим данные на 10 разделов и сохраним в формате parquete

In [7]:
%%time
df.count()

CPU times: user 1.15 ms, sys: 4.29 ms, total: 5.45 ms
Wall time: 21.7 s


101230332

In [8]:
(
    df
        .repartition(10)
        .write
        .mode("overwrite")
        .parquet("data/train.parquet")
)

Проверим размер сохраненного файла

In [9]:
!hdfs dfs -ls -h data/train.parquet

Found 11 items
-rw-r--r--   1 ubuntu hadoop          0 2025-02-11 11:08 data/train.parquet/_SUCCESS
-rw-r--r--   1 ubuntu hadoop    181.0 M 2025-02-11 11:07 data/train.parquet/part-00000-4cb2ab5a-66d5-40ec-b3d9-477526d9b41b-c000.snappy.parquet
-rw-r--r--   1 ubuntu hadoop    180.9 M 2025-02-11 11:07 data/train.parquet/part-00001-4cb2ab5a-66d5-40ec-b3d9-477526d9b41b-c000.snappy.parquet
-rw-r--r--   1 ubuntu hadoop    181.2 M 2025-02-11 11:07 data/train.parquet/part-00002-4cb2ab5a-66d5-40ec-b3d9-477526d9b41b-c000.snappy.parquet
-rw-r--r--   1 ubuntu hadoop    181.1 M 2025-02-11 11:07 data/train.parquet/part-00003-4cb2ab5a-66d5-40ec-b3d9-477526d9b41b-c000.snappy.parquet
-rw-r--r--   1 ubuntu hadoop    181.1 M 2025-02-11 11:07 data/train.parquet/part-00004-4cb2ab5a-66d5-40ec-b3d9-477526d9b41b-c000.snappy.parquet
-rw-r--r--   1 ubuntu hadoop    181.2 M 2025-02-11 11:07 data/train.parquet/part-00005-4cb2ab5a-66d5-40ec-b3d9-477526d9b41b-c000.snappy.parquet
-rw-r--r--   1 ubuntu hadoop    181.

Сравним скорость посчета строк из parquet

In [10]:
df_from_parquet = spark.read.parquet("data/train.parquet")

In [11]:
%%time
df_from_parquet.count()

CPU times: user 0 ns, sys: 2.58 ms, total: 2.58 ms
Wall time: 3.6 s


101230332